# Model Explainability — SHAP & LIME

**"Why did the model predict this?"** — the most critical question in ML.

- **SHAP** (SHapley Additive exPlanations): Game-theory based, globally consistent
- **LIME** (Local Interpretable Model-agnostic Explanations): Local linear approximations
- **Feature Importance**: Permutation, Impurity, SHAP-based
- **Partial Dependence Plots**: How a feature affects predictions
- **When to use which**: SHAP vs LIME comparison

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.preprocessing import StandardScaler

try:
    import shap
    print(f'SHAP: {shap.__version__}')
except ImportError:
    print('Install: pip install shap')

try:
    import lime
    import lime.lime_tabular
    print(f'LIME: {lime.__version__}')
except ImportError:
    print('Install: pip install lime')

import warnings
warnings.filterwarnings('ignore')

## 1. Train Models for Explanation

We'll train:
- **Classification**: RandomForest on Breast Cancer dataset
- **Regression**: GradientBoosting on California Housing

In [ ]:
# Classification setup
cancer = load_breast_cancer()
X_cls, y_cls = cancer.data, cancer.target
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_cls_train, y_cls_train)
print(f'Classification Accuracy: {clf.score(X_cls_test, y_cls_test):.4f}')

# Regression setup
housing = fetch_california_housing(as_frame=True)
X_reg, y_reg = housing.data, housing.target
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
reg = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
reg.fit(X_reg_train, y_reg_train)
print(f'Regression R² Score: {reg.score(X_reg_test, y_reg_test):.4f}')

## 2. Feature Importance — Three Methods Compared

In [ ]:
# Method 1: Impurity-based (built-in)
impurity_imp = clf.feature_importances_

# Method 2: Permutation importance
perm_imp = permutation_importance(clf, X_cls_test, y_cls_test, 
                                   n_repeats=10, random_state=42)

# Compare
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
sorted_idx = np.argsort(impurity_imp)
top_n = 15

axes[0].barh(range(top_n), impurity_imp[sorted_idx[-top_n:]], color='steelblue')
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(np.array(cancer.feature_names)[sorted_idx[-top_n:]])
axes[0].set_title('Impurity-Based Importance', fontsize=13)
axes[0].set_xlabel('Importance')

sorted_idx_perm = np.argsort(perm_imp.importances_mean)
axes[1].barh(range(top_n), perm_imp.importances_mean[sorted_idx_perm[-top_n:]], 
             color='coral', xerr=perm_imp.importances_std[sorted_idx_perm[-top_n:]])
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(np.array(cancer.feature_names)[sorted_idx_perm[-top_n:]])
axes[1].set_title('Permutation Importance', fontsize=13)
axes[1].set_xlabel('Mean Accuracy Decrease')

plt.suptitle('Feature Importance: Impurity vs Permutation', fontsize=14)
plt.tight_layout()
plt.show()

## 3. SHAP — Global & Local Explanations

SHAP values answer: **"How much did each feature contribute to this specific prediction?"**

Properties:
- **Local accuracy**: SHAP values sum to the prediction
- **Consistency**: If a feature contributes more, its SHAP value should be higher
- **Additivity**: `prediction = base_value + sum(SHAP values)`

In [ ]:
# SHAP for Classification (TreeExplainer — fast for tree models)
explainer_cls = shap.TreeExplainer(clf)
shap_values_cls = explainer_cls.shap_values(X_cls_test)

# Summary plot: Global feature importance
print('SHAP Summary Plot — Feature Impact on Predictions')
shap.summary_plot(shap_values_cls[1], X_cls_test, feature_names=cancer.feature_names,
                  show=True, max_display=15)

In [ ]:
# SHAP bar plot: Mean absolute SHAP values
shap.summary_plot(shap_values_cls[1], X_cls_test, feature_names=cancer.feature_names,
                  plot_type='bar', max_display=15, show=True)

In [ ]:
# Local explanation: Single prediction (Waterfall plot)
sample_idx = 0
print(f'Explaining prediction for sample {sample_idx}:')
print(f'  Actual label: {"Benign" if y_cls_test.iloc[sample_idx] if hasattr(y_cls_test, "iloc") else y_cls_test[sample_idx] else "Malignant"}')
print(f'  Predicted: {"Benign" if clf.predict(X_cls_test[sample_idx:sample_idx+1])[0] else "Malignant"}')

shap.initjs()
shap.force_plot(explainer_cls.expected_value[1], shap_values_cls[1][sample_idx],
               X_cls_test[sample_idx], feature_names=cancer.feature_names, matplotlib=True)

In [ ]:
# SHAP for Regression
explainer_reg = shap.TreeExplainer(reg)
shap_values_reg = explainer_reg.shap_values(X_reg_test)

shap.summary_plot(shap_values_reg, X_reg_test, feature_names=housing.feature_names,
                  max_display=10, show=True)

In [ ]:
# SHAP Dependence Plot — How one feature interacts with another
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
shap.dependence_plot('MedInc', shap_values_reg, X_reg_test, 
                     interaction_index='AveRooms', ax=axes[0], show=False)
axes[0].set_title('SHAP Dependence: MedInc (colored by AveRooms)')

shap.dependence_plot('Latitude', shap_values_reg, X_reg_test,
                     interaction_index='Longitude', ax=axes[1], show=False)
axes[1].set_title('SHAP Dependence: Latitude (colored by Longitude)')
plt.tight_layout()
plt.show()

## 4. LIME — Local Interpretable Explanations

LIME explains **individual predictions** by:
1. Perturbing the input data around the instance
2. Fitting a simple (interpretable) model locally
3. Showing which features pushed the prediction up/down

In [ ]:
# LIME for Classification
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_cls_train,
    feature_names=cancer.feature_names,
    class_names=['Malignant', 'Benign'],
    mode='classification'
)

# Explain a single prediction
sample_idx = 5
lime_exp = lime_explainer.explain_instance(
    X_cls_test[sample_idx], clf.predict_proba, num_features=10
)

print(f'LIME Explanation for Sample {sample_idx}')
print(f'Prediction: {clf.predict(X_cls_test[sample_idx:sample_idx+1])[0]}')
lime_exp.as_pyplot_figure()
plt.title('LIME — Local Feature Contributions')
plt.tight_layout()
plt.show()

In [ ]:
# LIME for Regression
lime_reg_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_reg_train.values,
    feature_names=housing.feature_names,
    mode='regression'
)

lime_reg_exp = lime_reg_explainer.explain_instance(
    X_reg_test.values[0], reg.predict, num_features=8
)

lime_reg_exp.as_pyplot_figure()
plt.title('LIME — Regression Feature Contributions')
plt.tight_layout()
plt.show()

## 5. Partial Dependence Plots

In [ ]:
# Partial Dependence Plots (sklearn built-in)
fig, ax = plt.subplots(figsize=(16, 6))
PartialDependenceDisplay.from_estimator(
    reg, X_reg_test, features=['MedInc', 'AveRooms', 'HouseAge', ('MedInc', 'AveRooms')],
    kind='average', ax=ax, n_cols=4
)
plt.suptitle('Partial Dependence Plots — California Housing', fontsize=14)
plt.tight_layout()
plt.show()

## 6. SHAP vs LIME — When to Use Which?

| Aspect | SHAP | LIME |
|--------|------|------|
| **Type** | Global + Local | Local only |
| **Theory** | Game theory (Shapley values) | Local linear approximation |
| **Consistency** | Mathematically guaranteed | No guarantees |
| **Speed** | TreeExplainer is fast; KernelExplainer is slow | Fast (perturbation-based) |
| **Model Agnostic?** | KernelSHAP yes, TreeSHAP no | Yes |
| **Best For** | Tree models, global understanding | Quick local explanations |
| **Visualization** | Excellent (summary, dependence, force) | Good (bar chart) |
| **Use In Production** | Yes (consistent, reliable) | Yes (explain individual predictions) |

### Recommendation
- Use **SHAP** as your primary explainability tool (more reliable, global + local)
- Use **LIME** for quick sanity checks or when SHAP is too slow (non-tree models)
- Use **Permutation Importance** for a fast, model-agnostic global overview
- Use **Partial Dependence Plots** to understand feature effects